# 08 - Comparaison Globale

## 1. Validation du protocole et resume des artefacts

Ce notebook charge uniquement des artefacts valides pour la synthese finale. Il ne formule pas de
conclusion; il assemble des tableaux et figures report-ready a partir des sorties confirmatoires.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.base import clone

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

sns.set_theme(style="whitegrid", context="talk")
pd.options.display.float_format = lambda value: f"{value:.4f}"

from src import (
    calculer_importance_permutation_exploratoire,
    calculer_metriques_classes_depuis_predictions,
    calculer_top_k_accuracies,
    charger_donnees,
    construire_matrice_confusion,
    construire_tableau_gaps,
    construire_tableau_plis_externes,
    construire_tableau_variantes,
    creer_notebook_context,
    evaluer_variantes_untuned,
    executer_tuning_confirmatoire,
    tracer_courbes_precision_rappel_ovr,
    tracer_heatmap_tuning,
    tracer_histogramme_confiance,
    tracer_matrice_confusion,
    tracer_metrique_par_classe,
    tracer_reliability_diagram,
    tracer_top_confusions,
)
from src.evaluation import (
    calculer_metriques,
    calculer_statistiques_confiance,
    evaluer_dummy_classifier,
    extraire_top_confusions,
)
from src.notebook_support import NOTEBOOK_MODEL_SPECS
from src.resultats import (
    MetricsArtifactValidationError,
    charger_tableau_mesures_valides,
    filtrer_legacy,
    normaliser_stades,
)


In [ ]:
def tracer_dot_whisker(tableau, categorie, moyenne, dispersion, titre, orientation="vertical"):
    donnees = tableau.sort_values(moyenne, ascending=(orientation == "horizontal")).copy()
    figure, axe = plt.subplots(figsize=(10.5, 5.8))
    if orientation == "horizontal":
        axe.errorbar(
            donnees[moyenne],
            donnees[categorie],
            xerr=donnees[dispersion],
            fmt="o",
            color="#0b1f3a",
            ecolor="#4C78A8",
            capsize=4,
            markersize=7,
        )
        axe.set_xlabel(moyenne.replace("_", " "), fontweight="bold")
        axe.set_ylabel(categorie.replace("_", " "), fontweight="bold")
    else:
        axe.errorbar(
            donnees[categorie],
            donnees[moyenne],
            yerr=donnees[dispersion],
            fmt="o",
            color="#0b1f3a",
            ecolor="#4C78A8",
            capsize=4,
            markersize=7,
        )
        axe.set_xlabel(categorie.replace("_", " "), fontweight="bold")
        axe.set_ylabel(moyenne.replace("_", " "), fontweight="bold")
        axe.tick_params(axis="x", rotation=30)
    axe.set_title(titre)
    sns.despine(ax=axe)
    figure.tight_layout()
    return axe


def tracer_barres_stades(tableau, titre):
    figure, axe = plt.subplots(figsize=(10.8, 6.0))
    sns.barplot(
        data=tableau,
        x="modele",
        y="val_f1_macro_mean",
        hue="stade",
        palette="crest",
        ax=axe,
    )
    axe.set_title(titre)
    axe.set_xlabel("Modele", fontweight="bold")
    axe.set_ylabel("Validation F1-Macro", fontweight="bold")
    axe.tick_params(axis="x", rotation=30)
    sns.move_legend(axe, "upper left", bbox_to_anchor=(1.02, 1), frameon=False, title="Stade")
    sns.despine(ax=axe)
    figure.tight_layout()
    return axe


def tracer_barres_gains(tableau, colonne, titre):
    donnees = tableau.sort_values(colonne, ascending=False).copy()
    figure, axe = plt.subplots(figsize=(10.2, 5.6))
    sns.barplot(data=donnees, x="modele", y=colonne, palette="mako", ax=axe)
    axe.axhline(0.0, color="#222222", linewidth=1.0)
    axe.set_title(titre)
    axe.set_xlabel("Modele", fontweight="bold")
    axe.set_ylabel(colonne.replace("_", " "), fontweight="bold")
    axe.tick_params(axis="x", rotation=30)
    sns.despine(ax=axe)
    figure.tight_layout()
    return axe


def tracer_points_plis(tableau, colonne, titre):
    donnees = tableau.sort_values("fold").copy()
    figure, axe = plt.subplots(figsize=(9.6, 5.4))
    sns.pointplot(data=donnees, x="fold", y=colonne, color="#0b1f3a", ax=axe)
    axe.set_title(titre)
    axe.set_xlabel("Pli externe", fontweight="bold")
    axe.set_ylabel(colonne.replace("_", " "), fontweight="bold")
    sns.despine(ax=axe)
    figure.tight_layout()
    return axe


In [ ]:
X, y, label_encoder = charger_donnees()

try:
    mesures = charger_tableau_mesures_valides(
        exclure_legacy=True,
        modeles=sorted(NOTEBOOK_MODEL_SPECS.keys()),
    )
except MetricsArtifactValidationError as exc:
    raise RuntimeError(
        "Des artefacts stale ou incompatibles bloquent la synthese globale. "
        "Relancer les notebooks 02 a 07 avant d'executer cette comparaison."
    ) from exc

mesures = filtrer_legacy(normaliser_stades(mesures))
display(mesures.sort_values(["modele", "evaluation_stage", "variante"]).round(4))


## 2. Reference baseline

La baseline de reference provient d'un DummyClassifier evalue avec le meme protocole simple CV.
Elle sert de plancher commun pour les gains par modele.


In [ ]:
baseline_dummy = evaluer_dummy_classifier(X, y)
baseline_table = pd.DataFrame([baseline_dummy]).assign(
    selection_protocol="simple_cv",
    evaluation_stage="baseline",
    timing_policy="cv_n_jobs=1",
)
display(baseline_table.round(4))


## 3. Meilleur resultat confirmatoire par modele

Le classement principal repose sur les variantes tuned en nested CV, avec F1-macro comme metrique
principale et accuracy comme lecture secondaire.


In [ ]:
best_tuned = (
    mesures[mesures["evaluation_stage"] == "tuned"]
    .sort_values(["val_f1_macro_mean", "val_accuracy_mean"], ascending=False)
    .reset_index(drop=True)
)
display(best_tuned[[
    "modele",
    "variante",
    "selection_protocol",
    "val_f1_macro_mean",
    "val_f1_macro_std",
    "val_accuracy_mean",
    "val_accuracy_std",
    "elapsed_seconds",
]].round(4))

tracer_dot_whisker(
    best_tuned,
    categorie="modele",
    moyenne="val_f1_macro_mean",
    dispersion="val_f1_macro_std",
    titre="Classement confirmatoire des modeles",
    orientation="horizontal",
)
plt.show()


## 4. Decomposition par stades

Chaque modele est decompose en trois niveaux: baseline commune, meilleure variante non tuned, puis
resultat tuned confirmatoire.


In [ ]:
best_untuned = (
    mesures[mesures["evaluation_stage"] != "tuned"]
    .sort_values(["modele", "val_f1_macro_mean"], ascending=[True, False])
    .groupby("modele", as_index=False)
    .first()
)

stage_table = pd.concat(
    [
        pd.DataFrame(
            {
                "modele": best_tuned["modele"],
                "stade": "baseline",
                "val_f1_macro_mean": baseline_dummy["val_f1_macro_mean"],
            }
        ),
        best_untuned[["modele", "val_f1_macro_mean"]].assign(stade="best_untuned"),
        best_tuned[["modele", "val_f1_macro_mean"]].assign(stade="tuned"),
    ],
    ignore_index=True,
)
display(stage_table.round(4))
tracer_barres_stades(stage_table, titre="Baseline -> meilleur untuned -> tuned")
plt.show()


## 5. Stabilite et cout

Les vues suivantes documentent les gains versus baseline et versus meilleure variante non tuned, la
dispersion inter-plis des variantes tuned, puis le couple temps/performance lorsque la timing policy
est uniforme.


In [ ]:
gains = (
    best_tuned[["modele", "val_f1_macro_mean", "val_f1_macro_std", "elapsed_seconds"]]
    .merge(
        best_untuned[["modele", "val_f1_macro_mean"]].rename(
            columns={"val_f1_macro_mean": "best_untuned_f1"}
        ),
        on="modele",
        how="left",
    )
)
gains["gain_vs_dummy"] = gains["val_f1_macro_mean"] - baseline_dummy["val_f1_macro_mean"]
gains["gain_vs_best_untuned"] = gains["val_f1_macro_mean"] - gains["best_untuned_f1"]

display(gains.round(4))
tracer_barres_gains(gains, "gain_vs_dummy", "Gain en F1-macro versus DummyClassifier")
plt.show()
tracer_barres_gains(gains, "gain_vs_best_untuned", "Gain en F1-macro versus meilleur untuned")
plt.show()

stabilite = best_tuned[["modele", "val_f1_macro_std"]].sort_values("val_f1_macro_std")
figure, axe = plt.subplots(figsize=(9.8, 5.4))
sns.barplot(data=stabilite, x="modele", y="val_f1_macro_std", palette="crest", ax=axe)
axe.set_title("Stabilite des resultats tuned")
axe.set_xlabel("Modele", fontweight="bold")
axe.set_ylabel("Ecart-type F1-macro", fontweight="bold")
axe.tick_params(axis="x", rotation=30)
sns.despine(ax=axe)
figure.tight_layout()
plt.show()

if mesures["timing_policy"].nunique() == 1:
    figure, axe = plt.subplots(figsize=(10.0, 5.8))
    sns.scatterplot(
        data=best_tuned,
        x="elapsed_seconds",
        y="val_f1_macro_mean",
        hue="modele",
        s=120,
        ax=axe,
    )
    axe.set_title("Temps total versus performance tuned")
    axe.set_xlabel("Elapsed seconds", fontweight="bold")
    axe.set_ylabel("Validation F1-macro", fontweight="bold")
    sns.move_legend(axe, "upper left", bbox_to_anchor=(1.02, 1), frameon=False)
    sns.despine(ax=axe)
    figure.tight_layout()
    plt.show()


## 6. Diagnostics multiclasses du meilleur modele

Les diagnostics ci-dessous se concentrent sur le meilleur modele confirmatoire uniquement, avec les
classes les plus difficiles, les top confusions et la matrice de confusion normalisee en appendice.


In [ ]:
winner = best_tuned.iloc[0]["modele"]
predictions_path = ROOT / "results" / "predictions" / f"{winner}__tuned__predictions.csv"
if not predictions_path.exists():
    raise FileNotFoundError(
        f"Predictions OOF manquantes pour le meilleur modele: {predictions_path}"
    )

winner_predictions = pd.read_csv(predictions_path)
winner_confusions = extraire_top_confusions(
    winner_predictions["y_true"].to_numpy(),
    winner_predictions["y_pred"].to_numpy(),
    label_encoder=label_encoder,
    top_n=12,
)
winner_class_metrics = calculer_metriques_classes_depuis_predictions(
    winner_predictions,
    label_encoder=label_encoder,
)
winner_confusion_matrix = construire_matrice_confusion(
    winner_predictions["y_true"].to_numpy(),
    winner_predictions["y_pred"].to_numpy(),
    label_encoder=label_encoder,
    normalize="true",
)

display(pd.DataFrame([{"winner": winner, **calculer_metriques(winner_predictions["y_true"], winner_predictions["y_pred"])}]).round(4))
display(winner_confusions.round(4))
display(winner_class_metrics.sort_values(["recall", "f1"]).head(15).round(4))
tracer_top_confusions(winner_confusions, top_n=12, titre=f"Top confusions - {winner}")
plt.show()
tracer_metrique_par_classe(
    winner_class_metrics,
    metrique="recall",
    top_n=15,
    ordre="asc",
    titre=f"Classes les plus difficiles - {winner}",
)
plt.show()
tracer_matrice_confusion(
    winner_confusion_matrix,
    titre=f"Matrice de confusion normalisee - {winner}",
    annot=False,
    rotation_x=90,
)
plt.show()
